In [19]:
import torch
import open_clip

device = "cuda" if torch.cuda.is_available() else "cpu"

model, _, preprocess = open_clip.create_model_and_transforms(
    "ViT-B-32",
    pretrained="openai"
)

model = model.to(device)
model.eval()

CLIP(
  (visual): VisionTransformer(
    (conv1): Conv2d(3, 768, kernel_size=(32, 32), stride=(32, 32), bias=False)
    (patch_dropout): Identity()
    (ln_pre): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
    (transformer): Transformer(
      (resblocks): ModuleList(
        (0-11): 12 x ResidualAttentionBlock(
          (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (attn): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=768, out_features=768, bias=True)
          )
          (ls_1): Identity()
          (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (mlp): Sequential(
            (c_fc): Linear(in_features=768, out_features=3072, bias=True)
            (gelu): GELU(approximate='none')
            (c_proj): Linear(in_features=3072, out_features=768, bias=True)
          )
          (ls_2): Identity()
        )
      )
    )
    (ln_post): LayerNorm((768,), eps=1e-05, elementwise_affine

In [1]:
import torch
import open_clip
import torch.nn.functional as F
from PIL import Image

device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Using device: {device}")
# -------------------------------------------------
# 1. Load CLIP
# -------------------------------------------------
model, _, preprocess = open_clip.create_model_and_transforms(
    "ViT-B-32",
    pretrained="openai",
)

model = model.to(device)
model.eval()

# -------------------------------------------------
# 2. Storage for GMAR
# -------------------------------------------------
all_attn = []
all_grads = []

# -------------------------------------------------
# 3. Monkey-patch ONLY attention forward (NO hooks)
# -------------------------------------------------
def enable_attention_capture(model):

    for block in model.visual.transformer.resblocks:

        attn_module = block.attn

        original_forward = attn_module.forward

        def forward_with_capture(*args, **kwargs):

            # Force attention weights
            kwargs["need_weights"] = True
            kwargs["average_attn_weights"] = False

            # Run original attention
            out = original_forward(*args, **kwargs)

            # CLIP returns tuple: (output, attn)
            attn_output, attn_weights = out

            # Ensure gradient tracking on attention
            attn_weights = attn_weights.requires_grad_(True)
            attn_weights.retain_grad()

            all_attn.append(attn_weights)

            return attn_output, attn_weights

        attn_module.forward = forward_with_capture


# Enable capture
enable_attention_capture(model)

# -------------------------------------------------
# 4. Input preparation
# -------------------------------------------------
image_path = "/Users/sohamjoita5498gmail.com/Documents/prune/GMAR-Plus-with-pruning/images/t.jpeg"
pil_img = Image.open(image_path).convert("RGB")

# # CLIP (ViT-B-32) expects 224x224 inputs. If your image is smaller (e.g. 64x64),
# # upscale it with bicubic interpolation before applying the model's preprocess.
# # This preserves the expected input resolution for positional embeddings.
# TARGET_SIZE = 224
# if pil_img.size != (TARGET_SIZE, TARGET_SIZE):
#     pil_img = pil_img.resize((TARGET_SIZE, TARGET_SIZE), Image.BICUBIC)

image = preprocess(pil_img).unsqueeze(0).to(device)

texts = ["tree", "dog", "birds"]
tokenized = open_clip.tokenize(texts).to(device)

# -------------------------------------------------
# 5. Forward pass
# -------------------------------------------------
image_features = model.encode_image(image)
text_features = model.encode_text(tokenized)

logits = image_features @ text_features.T

# -------------------------------------------------
# 6. Select target (text-as-class)
# -------------------------------------------------
pred_class = logits.argmax(dim=-1).item()
target = logits[0, pred_class]

# -------------------------------------------------
# 7. Backward pass
# -------------------------------------------------
model.zero_grad()
target.backward()

# -------------------------------------------------
# 8. Collect gradients for GMAR
# -------------------------------------------------
for attn in all_attn:
    if attn.grad is not None:
        all_grads.append(attn.grad)

# -------------------------------------------------
# 9. Sanity check
# -------------------------------------------------
print("Layers captured:", len(all_attn))
print("Gradients captured:", len(all_grads))

for i in range(len(all_attn)):
    print(
        f"Layer {i} | "
        f"Attn: {all_attn[i].shape} | "
        f"Grad: {all_grads[i].shape if i < len(all_grads) else None}"
    )

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cpu


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/open_clip/factory.py:450: UserWarning: QuickGELU mismatch between final model config (quick_gelu=False) and pretrained tag 'openai' (quick_gelu=True).
  warnings.warn(


Layers captured: 12
Gradients captured: 0
Layer 0 | Attn: torch.Size([1, 12, 50, 50]) | Grad: None
Layer 1 | Attn: torch.Size([1, 12, 50, 50]) | Grad: None
Layer 2 | Attn: torch.Size([1, 12, 50, 50]) | Grad: None
Layer 3 | Attn: torch.Size([1, 12, 50, 50]) | Grad: None
Layer 4 | Attn: torch.Size([1, 12, 50, 50]) | Grad: None
Layer 5 | Attn: torch.Size([1, 12, 50, 50]) | Grad: None
Layer 6 | Attn: torch.Size([1, 12, 50, 50]) | Grad: None
Layer 7 | Attn: torch.Size([1, 12, 50, 50]) | Grad: None
Layer 8 | Attn: torch.Size([1, 12, 50, 50]) | Grad: None
Layer 9 | Attn: torch.Size([1, 12, 50, 50]) | Grad: None
Layer 10 | Attn: torch.Size([1, 12, 50, 50]) | Grad: None
Layer 11 | Attn: torch.Size([1, 12, 50, 50]) | Grad: None


In [2]:
import torch

# -----------------------------
# logits = image_features @ text_features.T
# -----------------------------

probs = logits.softmax(dim=-1)

pred_idx = probs.argmax(dim=-1).item()

print("Raw logits:", logits.detach().cpu())
print("Probabilities:", probs.detach().cpu())

print("\nPredicted class index:", pred_idx)
print("Predicted label:", texts[pred_idx])
print("Confidence:", probs[0, pred_idx].item())

Raw logits: tensor([[13.9782, 15.6561, 15.6038]])
Probabilities: tensor([[0.0874, 0.4682, 0.4444]])

Predicted class index: 1
Predicted label: dog
Confidence: 0.4681893289089203
